# Thermal & Radar Students (Colab)

Two INDEPENDENT students per the POC data plan (2026-08-19), trained on shards
from `perception/export/export_shards.py` uploaded to Drive. GPU runtime (T4).

- **Thermal Student**: 6 derived channels from the raw 160×120 frame → grade-A
  person boxes (thermal px). Fully runnable on export v1.
- **Radar Student**: the frame's point set → person presence + horizontal
  position *u* in RGB pixels, against the RGB Teacher's box centers. It learns
  points→u **directly**, so the radar↔camera extrinsic is absorbed by the
  network — this is what makes it runnable before H5 (the 10/15 m flank
  stations) is measured. H5 still matters for *hand-projected* overlays and
  for judging flank errors; it does not block this training.
- **Fusion**: deliberately absent — step 8 of the plan, after both students
  are understood alone.

Philosophy: preserve information first → let the model learn → measure what it
used (ablation) → only then decide what deserves hardware.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/thermal-fusion/gexport/v1'

In [ ]:
import json, os, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

with open(os.path.join(DATA, 'manifest.json')) as f:
    MAN = json.load(f)
print(json.dumps(MAN['split'], indent=1))
TH_W, TH_H = MAN['thermal']['w'], MAN['thermal']['h']
RGB_W = 640
RADAR_K = MAN['radar']['k']
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

def to_celsius_or_unit(th_u8, sess):
    """uint8 -> degrees C when the session recorded its window, else [0,1].
    Recording the window (live.py meta since 2026-08-19) is what makes
    sessions with DIFFERENT windows commensurable: convert both to C and the
    model sees one scale. v1 sessions predate it -> unit scale, one era only."""
    info = MAN['sessions'][sess]
    if info.get('c_per_lsb'):
        return th_u8.astype(np.float32) * info['c_per_lsb'] + info['tmin']
    return th_u8.astype(np.float32) / 255.0

## Shared loading (time-ordered per session)

In [ ]:
def load_split(which):
    """-> dict of concatenated arrays, session-contiguous, time-ordered."""
    out = {'thermal': [], 'dt_ms': [], 'radar': [], 'n_radar': [],
           'th_boxes': [], 'n_th_boxes': [], 'rgb_boxes': [], 'n_rgb_boxes': [],
           'scaled': []}
    for sess in MAN['split'][which]:
        for p in sorted(glob.glob(os.path.join(DATA, f'{sess}-*.npz'))):
            z = np.load(p)
            for k in out:
                if k == 'scaled':
                    out[k].append(to_celsius_or_unit(z['thermal'], sess))
                elif k in z:
                    out[k].append(z[k])
    return {k: np.concatenate(v) for k, v in out.items() if v}

## Thermal Student

| # | channel | note |
|---|---------|------|
| 0 | intensity / temperature | scaled per session |
| 1–2 | grad-x, grad-y | |
| 3 | grad magnitude | |
| 4 | Δ from frame median | |
| 5 | temporal diff | zeroed when dt_ms>300 (FFC gap ≠ motion) |

In [ ]:
N_CH = 6
CH_NAMES = ['intensity', 'grad_x', 'grad_y', 'grad_mag', 'delta_bg', 'temporal']
STRIDE = 4
HM_W, HM_H = TH_W // STRIDE, TH_H // STRIDE

def build_channels(t, dt_ms):
    gx = np.gradient(t, axis=2); gy = np.gradient(t, axis=1)
    gmag = np.sqrt(gx**2 + gy**2)
    dbg = t - np.median(t, axis=(1, 2), keepdims=True)
    tdiff = np.zeros_like(t)
    ok = (dt_ms[1:] > 0) & (dt_ms[1:] < 300.0)
    tdiff[1:][ok] = t[1:][ok] - t[:-1][ok]
    return np.stack([t, gx, gy, gmag, dbg, tdiff], axis=1)

def make_targets(th_boxes, n_boxes):
    N = len(n_boxes)
    hm = np.zeros((N, 1, HM_H, HM_W), np.float32)
    wh = np.zeros((N, 2, HM_H, HM_W), np.float32)
    mask = np.zeros((N, 1, HM_H, HM_W), np.float32)
    yy, xx = np.mgrid[0:HM_H, 0:HM_W]
    for i in range(N):
        for b in range(int(n_boxes[i])):
            x, y, w, h = th_boxes[i, b, :4]
            cx, cy = int((x + w/2)/STRIDE), int((y + h/2)/STRIDE)
            if 0 <= cx < HM_W and 0 <= cy < HM_H:
                s = max(1.0, (w + h)/(4*STRIDE))
                hm[i, 0] = np.maximum(hm[i, 0], np.exp(-((xx-cx)**2 + (yy-cy)**2)/(2*s*s)))
                wh[i, :, cy, cx] = [w/TH_W, h/TH_H]
                mask[i, 0, cy, cx] = 1
    return hm, wh, mask

class ThermalStudent(nn.Module):
    def __init__(self, n_ch=N_CH):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(n_ch, 32, 3, 2, 1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU())
        self.head_hm = nn.Conv2d(64, 1, 1)
        self.head_wh = nn.Conv2d(64, 2, 1)
    def forward(self, x):
        f = self.backbone(x)
        return self.head_hm(f), self.head_wh(f)

In [ ]:
def eval_thermal(model, X, hm, wh, mask, bs=64):
    """Hit-rate: predicted heatmap peak within 2 cells of a true center."""
    model.eval(); hits = tot = 0
    with torch.no_grad():
        for i in range(0, len(X), bs):
            hp, _ = model(torch.from_numpy(X[i:i+bs]).to(DEV))
            hp = torch.sigmoid(hp).cpu().numpy()
            for j in range(len(hp)):
                m = mask[i+j, 0]
                if m.sum() == 0: continue
                tot += 1
                py, px = np.unravel_index(hp[j, 0].argmax(), hp[j, 0].shape)
                ty, tx = np.where(m > 0)
                if np.min(np.hypot(ty-py, tx-px)) <= 2: hits += 1
    return hits / max(tot, 1)

def train_thermal(channel_mask=None, epochs=30, bs=64, tag='thermal-full'):
    tr, va = load_split('train'), load_split('val')
    Xt = build_channels(tr['scaled'], tr['dt_ms'])
    Xv = build_channels(va['scaled'], va['dt_ms'])
    if channel_mask is not None:   # ablation: zero the channel, keep the slot
        m = np.asarray(channel_mask, np.float32)[None, :, None, None]
        Xt, Xv = Xt*m, Xv*m
    hmt, wht, mt = make_targets(tr['th_boxes'], tr['n_th_boxes'])
    hmv, whv, mv = make_targets(va['th_boxes'], va['n_th_boxes'])
    model = ThermalStudent().to(DEV)
    opt = torch.optim.AdamW(model.parameters(), 3e-4)
    order = np.arange(len(Xt))
    for ep in range(epochs):
        model.train(); np.random.shuffle(order)
        for i in range(0, len(order), bs):
            k = order[i:i+bs]
            hp, wp = model(torch.from_numpy(Xt[k]).to(DEV))
            hb = torch.from_numpy(hmt[k]).to(DEV)
            wb = torch.from_numpy(wht[k]).to(DEV)
            mb = torch.from_numpy(mt[k]).to(DEV)
            loss = (F.binary_cross_entropy_with_logits(hp, hb)
                    + 0.1*(F.l1_loss(wp, wb, reduction='none')*mb).sum()/mb.sum().clamp(1))
            opt.zero_grad(); loss.backward(); opt.step()
        if (ep+1) % 10 == 0:
            print(f'[{tag}] ep {ep+1}: val hit-rate {eval_thermal(model, Xv, hmv, whv, mv):.3f}')
    acc = eval_thermal(model, Xv, hmv, whv, mv)
    print(f'[{tag}] FINAL {acc:.3f}')
    return model, acc

thermal_model, thermal_full = train_thermal()

In [ ]:
# Thermal ablation - retrain from scratch per removed channel ('could the
# model do without it?', not 'does this trained model break without it?')
thermal_results = {'full': thermal_full}
for ci, name in enumerate(CH_NAMES):
    m = [1.0]*N_CH; m[ci] = 0.0
    _, acc = train_thermal(channel_mask=m, tag=f'thermal-no-{name}')
    thermal_results[f'no-{name}'] = acc
print('\n=== thermal ablation ===')
for k, v in sorted(thermal_results.items(), key=lambda kv: -kv[1]):
    print(f'{k:16s} {v:.3f}  (drop {thermal_results["full"]-v:+.3f})')

## Radar Student

Points (K=64, SNR-ranked; |v| only — folded velocity's sign is untrusted) →
two heads: person **presence** this frame, and horizontal position **u/RGB_W**
of the teacher's best box center. Frames where the teacher saw no one are the
negatives — they matter as much as the positives.

The u-target lives in RGB pixel space; the network learns the radar→pixel
mapping implicitly (the extrinsic is absorbed), so H5's flank residual does
not gate this — it will show up, if it matters, as higher u-error on flank
frames, which the eval below reports by |u| bucket.

In [ ]:
def radar_xy(split):
    R = split['radar'].copy(); R[..., 3] = np.abs(R[..., 3])
    n = split['n_radar'].astype(np.int64)
    nb = split['n_rgb_boxes']; bx = split['rgb_boxes']
    present = (nb > 0).astype(np.float32)
    u = np.zeros(len(nb), np.float32)
    for i in range(len(nb)):
        if nb[i] > 0:
            b = bx[i, :int(nb[i])]
            j = int(np.argmax(b[:, 4]))            # highest-confidence person
            u[i] = (b[j, 0] + b[j, 2]/2) / RGB_W
    return R, n, present, u

class RadarStudent(nn.Module):
    def __init__(self, feat=64):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(6, 64), nn.ReLU(), nn.Linear(64, feat))
        self.head = nn.Sequential(nn.Linear(feat, 64), nn.ReLU(), nn.Linear(64, 2))
    def forward(self, pts, n_pts):
        f = self.mlp(pts)
        idx = torch.arange(pts.shape[1], device=pts.device)[None, :]
        valid = (idx < n_pts[:, None]).unsqueeze(-1)
        f = f.masked_fill(~valid, -1e9).max(1).values
        f = f.masked_fill((n_pts == 0)[:, None], 0.0)
        o = self.head(f)
        return o[:, 0], torch.sigmoid(o[:, 1])     # presence logit, u

In [ ]:
def train_radar(feature_mask=None, epochs=40, bs=256, tag='radar-full'):
    """feature_mask zeroes point columns [x,y,z,v,snr,noise] for ablation."""
    tr, va = load_split('train'), load_split('val')
    Rt, nt, pt_, ut = radar_xy(tr); Rv, nv, pv, uv = radar_xy(va)
    if feature_mask is not None:
        m = np.asarray(feature_mask, np.float32)[None, None, :]
        Rt, Rv = Rt*m, Rv*m
    model = RadarStudent().to(DEV)
    opt = torch.optim.AdamW(model.parameters(), 1e-3)
    order = np.arange(len(Rt))
    for ep in range(epochs):
        model.train(); np.random.shuffle(order)
        for i in range(0, len(order), bs):
            k = order[i:i+bs]
            pl, uo = model(torch.from_numpy(Rt[k]).to(DEV),
                           torch.from_numpy(nt[k]).to(DEV))
            pb = torch.from_numpy(pt_[k]).to(DEV)
            ub = torch.from_numpy(ut[k]).to(DEV)
            loss = F.binary_cross_entropy_with_logits(pl, pb) \
                 + (torch.abs(uo - ub) * pb).sum() / pb.sum().clamp(1)
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        pl, uo = model(torch.from_numpy(Rv).to(DEV), torch.from_numpy(nv).to(DEV))
        pred = (torch.sigmoid(pl).cpu().numpy() > 0.5)
        acc = float((pred == (pv > 0.5)).mean())
        upx = np.abs(uo.cpu().numpy() - uv)[pv > 0.5] * RGB_W
        # error by |u - center|: flank frames degraded by the H5 residual
        # separate from center frames, so H5's fix is measurable here later
        cen = np.abs(uv[pv > 0.5] - 0.5) < 0.2
        print(f'[{tag}] presence acc {acc:.3f}; u-err px: '
              f'center {np.median(upx[cen]) if cen.any() else float("nan"):.0f}, '
              f'flank {np.median(upx[~cen]) if (~cen).any() else float("nan"):.0f}, '
              f'all {np.median(upx) if len(upx) else float("nan"):.0f}')
    return model, acc

radar_model, radar_full = train_radar()

# radar ablation over point features
radar_results = {'full': radar_full}
for ci, name in enumerate(['x', 'y', 'z', 'v', 'snr', 'noise']):
    m = [1.0]*6; m[ci] = 0.0
    _, acc = train_radar(feature_mask=m, tag=f'radar-no-{name}')
    radar_results[f'no-{name}'] = acc
print('\n=== radar ablation ===')
for k, v in sorted(radar_results.items(), key=lambda kv: -kv[1]):
    print(f'{k:12s} {v:.3f}  (drop {radar_results["full"]-v:+.3f})')

## Reading the results

- **Thermal**: a channel whose removal costs nothing was either learned
  internally (fine for derived channels) or unexercised by this dataset —
  collect the hard sessions (7–15 m, clutter) before believing a null.
- **Radar**: v1 is indoor and short-range; expect Doppler (`v`) to matter
  little on stationary holds and much more on the walking campaign data.
  The flank-vs-center u-error gap is the H5 residual made measurable — after
  the 10/15 m TCR stations, retrain and watch that gap close.
- Save weights: `torch.save(model.state_dict(), os.path.join(DATA, '<name>.pt'))`.
- Future shards with RA heatmaps (radar_people_ra.cfg sessions) plug in as a
  2-D radar input; the point-based student above stays as the baseline.